In [ ]:

from pyspark.sql import SparkSession, functions as F
from pathlib import Path
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType,
    DecimalType, TimestampType
)

spark = (
    SparkSession.builder
    .appName("FestivalDataDetective")
    .master("local[*]")
    .getOrCreate()
)


In [ ]:
DATA_DIR = Path("Data/raw")

files = {
    'attendees': "attendees.csv",
    'food_orders': "food_orders.csv",
    'food_stalls': "food_stalls.csv",
    'incidents': "incidents.csv",
    'performances': "performances.csv",
    "stages": "stages.csv",
    "ticket_scans": "ticket_scans.csv"  
}
raw = {}

for name, filename in files.items():
    file_path = DATA_DIR /filename
    raw[name] = (
        spark.read
        .option("header", True)             
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(str(file_path))
    )

print("Successfully loaded", len(raw), "datasets")

In [ ]:
for name, df in raw.items():
    print('\n',"=" * 70)
    print(f'DATASET: {name.upper()}')
    print(f'Rows: {df.count():,}')
    print(f'Columns({len(df.columns)}): {df.columns}')
    print("-"*70)
    df.printSchema()
    df.show(3,truncate=False)

In [ ]:

schemas = {
    "attendees": StructType([
        StructField("attendee_id", StringType(), True),
        StructField("ticket_type", StringType(), True),
        StructField("age_group", StringType(), True),
        StructField("home_city", StringType(), True),
    ]),

    "food_orders": StructType([
        StructField("order_id", StringType(), True),
        StructField("attendee_id", StringType(), True),
        StructField("stall_id", StringType(), True),
        StructField("order_time", TimestampType(), True),
        StructField("amount", DecimalType(10, 2), True),
        StructField("order_status", StringType(), True),
    ]),

    "food_stalls": StructType([
        StructField("stall_id", StringType(), True),
        StructField("stall_name", StringType(), True),
        StructField("cuisine", StringType(), True),
        StructField("stage_id", StringType(), True),
    ]),

    "incidents": StructType([
        StructField("incident_id", StringType(), True),
        StructField("stage_id", StringType(), True),
        StructField("incident_time", TimestampType(), True),
        StructField("incident_type", StringType(), True),
        StructField("severity", StringType(), True),
    ]),

    "performances": StructType([
        StructField("performance_id", StringType(), True),
        StructField("artist_name", StringType(), True),
        StructField("stage_id", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("genre", StringType(), True),
    ]),

    "stages": StructType([
        StructField("stage_id", StringType(), True),
        StructField("stage_name", StringType(), True),
        StructField("capacity", IntegerType(), True),
        StructField("zone", StringType(), True),
    ]),

    "ticket_scans": StructType([
        StructField("scan_id", StringType(), True),
        StructField("attendee_id", StringType(), True),
        StructField("stage_id", StringType(), True),
        StructField("scan_time", TimestampType(), True),
        StructField("scan_status", StringType(), True),
    ]),
}

print("Defined schemas:", len(schemas))


In [ ]:

typed = {}

for name, filename in files.items():
    typed[name] = (
        spark.read
        .schema(schemas[name])
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .csv(str(DATA_DIR / filename))
    )

print("Typed datasets:", len(typed))

for name, df in typed.items():
    print(f"\n{name.upper()}")
    df.printSchema()


In [ ]:

for name, df in typed.items():
    print(f"\n{'=' * 60}")
    print(f"DATASET: {name.upper()}")

    expressions = [
        F.count("*").alias("row_count")
    ]

    for column in df.columns:
        expressions.append(
            F.sum(
                F.when(
                    F.col(column).isNull()
                    | (F.trim(F.col(column).cast("string")) == ""),
                    1
                ).otherwise(0)
            ).alias(f"{column}_missing")
        )

    df.agg(*expressions).show(truncate=False)


In [ ]:


primary_keys = {
    "attendees": "attendee_id",
    "food_orders": "order_id",
    "food_stalls": "stall_id",
    "incidents": "incident_id",
    "performances": "performance_id",
    "stages": "stage_id",
    "ticket_scans": "scan_id",
}

for name, key in primary_keys.items():
    df = typed[name]

    duplicate_groups = (
        df.groupBy(key)
        .count()
        .filter(F.col(key).isNotNull() & (F.col("count") > 1))
    )

    print(f"\n{name.upper()} — duplicate {key} groups")
    duplicate_groups.show(20, truncate=False)

    print("Number of duplicate groups:", duplicate_groups.count())

In [ ]:

category_columns = {
    "attendees": ["ticket_type", "age_group", "home_city"],
    "food_orders": ["order_status"],
    "food_stalls": ["cuisine"],
    "incidents": ["incident_type", "severity"],
    "performances": ["genre"],
    "stages": ["zone"],
    "ticket_scans": ["scan_status"],
}

for name, columns in category_columns.items():
    print(f"\n{'=' * 55}")
    print(name.upper())

    for column in columns:
        print(f"\nValues in {column}:")

        (
            typed[name]
            .groupBy(F.lower(F.trim(F.col(column).cast("string"))).alias("normalized_value"))
            .count()
            .orderBy(F.desc("count"))
            .show(30, truncate=False)
        )


In [ ]:

print("FOOD ORDERS — amount distribution")
typed["food_orders"].select(
    F.min("amount").alias("min_amount"),
    F.max("amount").alias("max_amount"),
    F.sum(F.when(F.col("amount") < 0, 1).otherwise(0)).alias("negative_amounts"),
    F.sum(F.when(F.col("amount") == 0, 1).otherwise(0)).alias("zero_amounts"),
).show()

print("STAGES — capacity distribution")
typed["stages"].select(
    F.min("capacity").alias("min_capacity"),
    F.max("capacity").alias("max_capacity"),
    F.sum(F.when(F.col("capacity") <= 0, 1).otherwise(0)).alias("nonpositive_capacities"),
).show()

print("PERFORMANCES — invalid time ranges")
typed["performances"].filter(
    F.col("start_time").isNotNull()
    & F.col("end_time").isNotNull()
    & (F.col("end_time") <= F.col("start_time"))
).show(truncate=False)


In [ ]:
# 1. Inspect repeated food order IDs
print("DUPLICATE FOOD ORDERS")

duplicate_order_ids = (
    typed["food_orders"]
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
)

(
    typed["food_orders"]
    .join(duplicate_order_ids, on="order_id", how="inner")
    .orderBy("order_id")
    .show(truncate=False)
)

# 2. Inspect repeated scan IDs
print("DUPLICATE TICKET SCANS")

duplicate_scan_ids = (
    typed["ticket_scans"]
    .groupBy("scan_id")
    .count()
    .filter(F.col("count") > 1)
    .select("scan_id")
)

(
    typed["ticket_scans"]
    .join(duplicate_scan_ids, on="scan_id", how="inner")
    .orderBy("scan_id")
    .show(truncate=False)
)

# 3. Inspect negative food amounts
print("NEGATIVE FOOD AMOUNTS")

(
    typed["food_orders"]
    .filter(F.col("amount") < 0)
    .show(truncate=False)
)


In [ ]:

def show_orphan_records(
    source_df,
    source_key,
    target_df,
    target_key,
    relationship_name
):
    target_keys = (
        target_df
        .select(F.col(target_key).alias("_target_key"))
        .filter(F.col("_target_key").isNotNull())
        .distinct()
    )

    orphans = (
        source_df
        .filter(F.col(source_key).isNotNull())
        .join(
            target_keys,
            F.col(source_key) == F.col("_target_key"),
            "left_anti"
        )
    )

    print(f"\n{relationship_name}")
    print("Orphan rows:", orphans.count())
    orphans.show(10, truncate=False)


show_orphan_records(
    typed["food_orders"], "attendee_id",
    typed["attendees"], "attendee_id",
    "Food orders -> attendees"
)

show_orphan_records(
    typed["food_orders"], "stall_id",
    typed["food_stalls"], "stall_id",
    "Food orders -> food stalls"
)

show_orphan_records(
    typed["ticket_scans"], "attendee_id",
    typed["attendees"], "attendee_id",
    "Ticket scans -> attendees"
)

show_orphan_records(
    typed["ticket_scans"], "stage_id",
    typed["stages"], "stage_id",
    "Ticket scans -> stages"
)

show_orphan_records(
    typed["performances"], "stage_id",
    typed["stages"], "stage_id",
    "Performances -> stages"
)

show_orphan_records(
    typed["incidents"], "stage_id",
    typed["stages"], "stage_id",
    "Incidents -> stages"
)


In [ ]:

# Expected primary keys for each dataset
primary_keys = {
    "attendees": "attendee_id",
    "food_orders": "order_id",
    "food_stalls": "stall_id",
    "incidents": "incident_id",
    "performances": "performance_id",
    "stages": "stage_id",
    "ticket_scans": "scan_id",
}

# Records with duplicate primary keys
duplicate_keys = {}

for name, key in primary_keys.items():
    duplicate_keys[name] = (
        typed[name]
        .groupBy(key)
        .count()
        .filter(F.col(key).isNotNull() & (F.col("count") > 1))
        .select(key)
    )

# Reference keys
attendee_keys = typed["attendees"].select(
    F.col("attendee_id").alias("_ref_id")
).distinct()

stall_keys = typed["food_stalls"].select(
    F.col("stall_id").alias("_ref_id")
).distinct()

stage_keys = typed["stages"].select(
    F.col("stage_id").alias("_ref_id")
).distinct()


def unmatched_nonnull(df, column, reference_keys):
    """Rows whose non-null reference key doesn't exist in the lookup."""
    return (
        df.filter(F.col(column).isNotNull())
        .join(
            reference_keys,
            F.col(column) == F.col("_ref_id"),
            "left_anti"
        )
    )


# Define dataset-specific invalid-row rules
invalid_rules = {
    "attendees": lambda df: (
        F.col("attendee_id").isNull()
        | (F.trim("attendee_id") == "")
    ),

    "food_orders": lambda df: (
        F.col("order_id").isNull()
        | (F.trim("order_id") == "")
        | F.col("order_time").isNull()
        | F.col("amount").isNull()
        | (F.col("amount") < 0)
        | F.col("attendee_id").isNull()
        | F.col("stall_id").isNull()
    ),

    "food_stalls": lambda df: (
        F.col("stall_id").isNull()
        | (F.trim("stall_id") == "")
    ),

    "incidents": lambda df: (
        F.col("incident_id").isNull()
        | (F.trim("incident_id") == "")
        | F.col("stage_id").isNull()
        | F.col("incident_time").isNull()
        | (F.trim("incident_type") == "")
    ),

    "performances": lambda df: (
        F.col("performance_id").isNull()
        | (F.trim("performance_id") == "")
        | F.col("stage_id").isNull()
        | F.col("start_time").isNull()
        | F.col("end_time").isNull()
        | (F.col("end_time") <= F.col("start_time"))
        | (F.trim("artist_name") == "")
    ),

    "stages": lambda df: (
        F.col("stage_id").isNull()
        | (F.trim("stage_id") == "")
        | F.col("capacity").isNull()
        | (F.col("capacity") <= 0)
    ),

    "ticket_scans": lambda df: (
        F.col("scan_id").isNull()
        | (F.trim("scan_id") == "")
        | F.col("attendee_id").isNull()
        | F.col("stage_id").isNull()
        | F.col("scan_time").isNull()
    ),
}

# Reference-integrity rules
reference_rules = {
    "food_orders": [
        ("attendee_id", attendee_keys),
        ("stall_id", stall_keys),
    ],
    "ticket_scans": [
        ("attendee_id", attendee_keys),
        ("stage_id", stage_keys),
    ],
    "performances": [
        ("stage_id", stage_keys),
    ],
    "incidents": [
        ("stage_id", stage_keys),
    ],
}

clean = {}
quarantine = {}

for name, df in typed.items():
    # Attach duplicate and base validation flags
    base_invalid = invalid_rules[name](df)

    duplicate_condition = F.lit(False)
    for key in [primary_keys[name]]:
        duplicate_condition = duplicate_condition | (
            F.col(key).isNotNull()
            & F.col(key).isin(
                # Use a join instead of collecting all keys to Python.
                # The join-based duplicate logic is applied below.
                []
            )
        )

    # First quarantine duplicate key groups using left-semi join
    key = primary_keys[name]
    duplicate_rows = df.join(
        duplicate_keys[name], on=key, how="left_semi"
    )

    nonduplicate_rows = df.join(
        duplicate_keys[name], on=key, how="left_anti"
    )

    # Mark base-invalid records
    base_bad = nonduplicate_rows.filter(base_invalid)
    remaining = nonduplicate_rows.filter(~base_invalid)

    # Collect invalid rows for this dataset without dropping them silently
    invalid_parts = [
        duplicate_rows.withColumn(
            "quarantine_reason", F.lit("duplicate_primary_key")
        ),
        base_bad.withColumn(
            "quarantine_reason", F.lit("required_field_or_value_invalid")
        ),
    ]

    # Apply foreign-key rules one by one
    for fk_column, reference_df in reference_rules.get(name, []):
        bad_fk = unmatched_nonnull(
            remaining, fk_column, reference_df
        )
        if bad_fk.limit(1).count() > 0:
            invalid_parts.append(
                bad_fk.withColumn(
                    "quarantine_reason",
                    F.lit(f"unmatched_reference:{fk_column}")
                )
            )
        remaining = remaining.join(
            reference_df,
            F.col(fk_column) == F.col("_ref_id"),
            "left_semi"
        ).drop("_ref_id")

    # Union quarantine records with reason labels
    quarantine[name] = invalid_parts[0]
    for part in invalid_parts[1:]:
        quarantine[name] = quarantine[name].unionByName(
            part, allowMissingColumns=True
        )

    clean[name] = remaining

print("Clean and quarantine DataFrames created.")


In [ ]:
for name in typed:
    print(f"\n{name.upper()}")
    print("Raw rows:", typed[name].count())
    print("Clean rows:", clean[name].count())
    print("Quarantined rows:", quarantine[name].count())

    quarantine[name].groupBy("quarantine_reason").count().show(
        truncate=False
    )

In [ ]:

primary_keys = {
    "attendees": "attendee_id",
    "food_orders": "order_id",
    "food_stalls": "stall_id",
    "incidents": "incident_id",
    "performances": "performance_id",
    "stages": "stage_id",
    "ticket_scans": "scan_id",
}

for name in typed:
    raw_count = typed[name].count()
    clean_count = clean[name].count()
    quarantine_count = quarantine[name].count()

    key = primary_keys[name]

    duplicate_groups = (
        clean[name]
        .groupBy(key)
        .count()
        .filter(F.col(key).isNotNull() & (F.col("count") > 1))
        .count()
    )

    print(
        f"{name:15} "
        f"raw={raw_count:4} "
        f"clean={clean_count:4} "
        f"quarantine={quarantine_count:3} "
        f"reconciles={raw_count == clean_count + quarantine_count} "
        f"duplicate_key_groups={duplicate_groups}"
    )

In [ ]:

(
    quarantine["food_orders"]
    .filter(
        F.col("quarantine_reason")
        == "required_field_or_value_invalid"
    )
    .select(
        "order_id",
        "attendee_id",
        "stall_id",
        "order_time",
        "amount",
        "order_status",
        "quarantine_reason"
    )
    .orderBy("order_id")
    .show(50, truncate=False)
)

In [ ]:
(
    quarantine["food_orders"]
    .filter(F.col("quarantine_reason") == "duplicate_primary_key")
    .select("order_id", "attendee_id", "stall_id", "order_time", "amount", "order_status")
    .orderBy("order_id")
    .show(truncate=False)
)

In [ ]:


food_orders_raw = typed["food_orders"]

# Identify IDs that occur more than once
duplicate_ids = (
    food_orders_raw
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .select("order_id")
)

# Check whether duplicate groups are identical across all columns
duplicate_records = food_orders_raw.join(
    duplicate_ids,
    on="order_id",
    how="inner"
)

duplicate_variants = (
    duplicate_records
    .dropDuplicates()
    .groupBy("order_id")
    .count()
)

duplicate_variants.show(truncate=False)

In [ ]:

from pyspark.sql.window import Window

food_orders_raw = typed["food_orders"]

order_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        F.col("order_time").desc_nulls_last(),
        F.col("amount").desc_nulls_last()
    )
)

food_orders_ranked = food_orders_raw.withColumn(
    "row_num",
    F.row_number().over(order_window)
)

food_orders_deduplicated = (
    food_orders_ranked
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [ ]:
print("Raw rows:", food_orders_raw.count())
print("Deduplicated rows:", food_orders_deduplicated.count())

(
    food_orders_deduplicated
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .show(truncate=False)
)